In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# set font
plt.rcParams['font.family'] = 'Book Antiqua'
# import seaborn as sns
import matplotlib.dates as mdates
import calendar
import warnings
warnings.filterwarnings("ignore")

In [4]:
# lear = pd.read_csv("lear_lmp_extra_day.csv")
# cing_lear = pd.read_csv("cing_lear_extra_day.csv")
lear = pd.read_csv("lear_gefcom.csv")
cing_lear = pd.read_csv("cing_lear_gefcom.csv")
dnn = pd.read_csv("dnn_gefcom.csv")
dnn = dnn.rename(columns={"value":"DNN_lmp"})


# df_merged = lear.copy()
df_merged = lear.merge(
    cing_lear[["timestamp", "multitask_lmp"]],
    on=["timestamp"],
    how="inner"
)

df_merged = df_merged.merge(
    dnn[["timestamp", "DNN_lmp"]],
    on=["timestamp"],
    how="inner")


# # df_merged = pd.read_csv("df_merged.csv", index_col=0)
df_merged = df_merged.drop(columns=['datetime'])
df_merged["timestamp"] = pd.to_datetime(df_merged["timestamp"])
df_merged = df_merged.drop(columns=["Unnamed: 0"])
df_merged = df_merged.rename(columns={"timestamp":"datetime"})
df_merged = df_merged.rename(columns={"Price":"true_lmp"})
df_merged = df_merged.rename(columns={"value":"Lear_lmp"})




In [5]:
df_merged 

,datetime,hour,Lear_lmp,true_lmp,date,multitask_lmp,DNN_lmp
0,2013-06-16 00:00:00,0,32.602796,34.02,2013-06-16,32.897229,34.520477
1,2013-06-16 01:00:00,1,28.765740,28.26,2013-06-16,29.090762,30.693792
2,2013-06-16 02:00:00,2,25.175344,24.82,2013-06-16,25.497532,28.686991
3,2013-06-16 03:00:00,3,23.792433,22.90,2013-06-16,24.148876,28.172661
4,2013-06-16 04:00:00,4,23.438528,21.59,2013-06-16,23.868722,27.752504
...,...,...,...,...,...,...,...
4435,2013-12-17 19:00:00,19,113.294493,113.92,2013-12-17,119.442217,129.952789
4436,2013-12-17 20:00:00,20,109.309661,107.26,2013-12-17,107.770813,120.685623
4437,2013-12-17 21:00:00,21,96.362564,89.02,2013-12-17,95.980939,112.714478
4438,2013-12-17 22:00:00,22,79.490264,85.40,2013-12-17,83.808257,101.996765


### Visualization

In [7]:
model_col_candidates = ["Lear_lmp", "multitask_lmp", "DNN_lmp"]
model_col = [c for c in model_col_candidates if c in df_merged.columns]
missing_cols = [c for c in model_col_candidates if c not in df_merged.columns]
if missing_cols:
    print(f"跳过缺失列: {missing_cols}")
print(f"可用模型列: {model_col}")

可用模型列: ['Lear_lmp', 'multitask_lmp', 'DNN_lmp']


In [10]:
def pi_simultaneous_band_mc(
    df_merged: pd.DataFrame,
    start_date: str,
    end_date: str,
    N: int = 3,
    conf: float = 0.95,
    true_col: str = "true_lmp",
    pred_cols=None,
    M: int = 50000,
    seed: int = 123,
    require_full_days: bool = True,
    jitter: float = 1e-10,
):
    """
    返回 pi_dict[model] -> DataFrame(datetime, true, mean, lower, upper)
    其中 lower/upper 是“联立带”（simultaneous band），通过 MC 校准 q 得到：
      q = quantile_conf( max_h |(Y_h - mu_h)/sd_h| )
      band: mu_h ± q * sd_h
    """

    df = df_merged.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values("datetime")

    if pred_cols is None:
        pred_cols = [c for c in df.columns if c not in ["datetime", true_col]]
    if not pred_cols:
        raise ValueError("pred_cols 为空：请传入模型预测列名列表。")

    df = df[["datetime", true_col, *pred_cols]].dropna()
    df["date"] = df["datetime"].dt.floor("D")
    df["hour"] = df["datetime"].dt.hour

    if require_full_days:
        counts = df.groupby("date")["hour"].nunique()
        df = df[df["date"].isin(counts[counts == 24].index)]

    def pivot_day_hour(col):
        return (
            df.pivot_table(index="date", columns="hour", values=col, aggfunc="first")
              .reindex(columns=list(range(24)))
        )

    Y_true = pivot_day_hour(true_col)

    start_date = pd.to_datetime(start_date).floor("D")
    end_date = pd.to_datetime(end_date).floor("D")
    target_days = Y_true.index[(Y_true.index >= start_date) & (Y_true.index <= end_date)]

    rng = np.random.default_rng(seed)
    day_index = Y_true.index.to_numpy()

    pi_dict = {}

    for model_col in pred_cols:
        Y_hat = pivot_day_hour(model_col)
        E = (Y_hat - Y_true).to_numpy()  # residual = pred - true

        rows = []

        for d in target_days:
            pos_arr = np.where(day_index == np.datetime64(d))[0]
            if len(pos_arr) == 0:
                continue
            pos = int(pos_arr[0])
            if pos < N:
                continue

            # Sigma estimate from past N days
            E_win = E[pos - N:pos, :]                # (N,24)
            Sigma_hat = (E_win.T @ E_win) / N        # (24,24)
            Sigma_hat = (Sigma_hat + Sigma_hat.T) / 2.0
            Sigma_hat += jitter * np.eye(24)

            mu = Y_hat.loc[d].to_numpy()             # (24,)
            true_vec = Y_true.loc[d].to_numpy()      # (24,)

            sd = np.sqrt(np.clip(np.diag(Sigma_hat), 0.0, None))  # (24,)

            # correlated sampling
            draws = rng.multivariate_normal(mean=mu, cov=Sigma_hat, size=M)  # (M,24)

            # calibrate q for simultaneous coverage across 24 hours
            z = (draws - mu) / (sd + 1e-15)
            Tmax = np.max(np.abs(z), axis=1)         # (M,)
            q = np.quantile(Tmax, conf)

            lower = mu - q * sd
            upper = mu + q * sd

            for h in range(24):
                dt = pd.Timestamp(d) + pd.Timedelta(hours=h)
                rows.append(
                    {
                        "datetime": dt,
                        "true": float(true_vec[h]),
                        "mean": float(mu[h]),
                        "lower": float(lower[h]),
                        "upper": float(upper[h]),
                    }
                )

        pi_dict[model_col] = pd.DataFrame(rows).sort_values("datetime").reset_index(drop=True)

    return pi_dict

In [11]:
def pi_quantile_mc(
    df_merged: pd.DataFrame,
    start_date: str,
    end_date: str,
    N: int = 14,
    true_col: str = "true_lmp",
    pred_cols=None,
    quantiles=None,
    M: int = 20000,
    seed: int = 123,
    require_full_days: bool = True,
    jitter: float = 1e-10,
):
    """ 
    使用与 pi_simultaneous_band_mc 类似的残差协方差 + 多元正态 MC 思路，
    但返回的是逐小时的分位数预测（pointwise quantile），而不是联立带。

    返回：
        q_dict[model] -> DataFrame(
            datetime, true,
            q10, q20, ..., q90   # 分别对应 0.1, 0.2, ..., 0.9 分位数
        )
    """

    if quantiles is None:
        quantiles = [0.1 * k for k in range(1, 10)]  # 0.1, ..., 0.9

    df = df_merged.copy()
    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values("datetime")

    if pred_cols is None:
        pred_cols = [c for c in df.columns if c not in ["datetime", true_col]]
    missing = [c for c in pred_cols if c not in df.columns]
    if missing:
        print(f"警告: 跳过 df_merged 中不存在的列: {missing}")
    pred_cols = [c for c in pred_cols if c in df.columns]
    if not pred_cols:
        raise ValueError("pred_cols 为空：请传入存在于 df_merged 中的模型预测列名列表。")

    df = df[["datetime", true_col, *pred_cols]].dropna()
    df["date"] = df["datetime"].dt.floor("D")
    df["hour"] = df["datetime"].dt.hour

    if require_full_days:
        counts = df.groupby("date")["hour"].nunique()
        df = df[df["date"].isin(counts[counts == 24].index)]

    def pivot_day_hour(col):
        return (
            df.pivot_table(index="date", columns="hour", values=col, aggfunc="first")
              .reindex(columns=list(range(24)))
        )

    Y_true = pivot_day_hour(true_col)

    start_date = pd.to_datetime(start_date).floor("D")
    end_date = pd.to_datetime(end_date).floor("D")
    target_days = Y_true.index[(Y_true.index >= start_date) & (Y_true.index <= end_date)]

    rng = np.random.default_rng(seed)
    day_index = Y_true.index.to_numpy()

    q_dict = {}

    for model_col in pred_cols:
        Y_hat = pivot_day_hour(model_col)
        E = (Y_hat - Y_true).to_numpy()  # residual = pred - true

        rows = []

        for d in target_days:
            pos_arr = np.where(day_index == np.datetime64(d))[0]
            if len(pos_arr) == 0:
                continue
            pos = int(pos_arr[0])
            if pos < 1:
                continue

            # Sigma estimate from past days (use up to N days of available history)
            n_hist = min(N, pos)
            E_win = E[pos - n_hist:pos, :]           # (n_hist,24)
            Sigma_hat = (E_win.T @ E_win) / n_hist   # (24,24)
            Sigma_hat = (Sigma_hat + Sigma_hat.T) / 2.0
            Sigma_hat += jitter * np.eye(24)

            mu = Y_hat.loc[d].to_numpy()             # (24,)
            true_vec = Y_true.loc[d].to_numpy()      # (24,)

            # correlated sampling
            draws = rng.multivariate_normal(mean=mu, cov=Sigma_hat, size=M)  # (M,24)

            # 对每个小时的预测分布，直接取 0.1-0.9 分位数（pointwise），不再做联立校准
            # qs 的形状：(len(quantiles), 24)
            qs = np.quantile(draws, quantiles, axis=0)

            for h in range(24):
                dt = pd.Timestamp(d) + pd.Timedelta(hours=h)
                row = {
                    "datetime": dt,
                    "true": float(true_vec[h]),
                }
                for qi, qv in enumerate(quantiles):
                    col_name = f"q{int(qv * 100):02d}"  # 如 q10, q20, ..., q90
                    row[col_name] = float(qs[qi, h])
                rows.append(row)

        q_cols = ["datetime", "true"] + [f"q{int(qv * 100):02d}" for qv in quantiles]
        if rows:
            q_dict[model_col] = pd.DataFrame(rows).sort_values("datetime").reset_index(drop=True)
        else:
            q_dict[model_col] = pd.DataFrame(columns=q_cols)

    return q_dict

In [8]:
# 配置：你可以在这里改模型列表和时间范围
quantiles = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
# tail_quantiles = [0.01, 0.025, 0.05, 0.95, 0.975, 0.99]

# quantiles = quantiles+tail_quantiles

In [53]:
df_merged

,datetime,hour,Lear_lmp,true_lmp,date,multitask_lmp
0,2013-06-16 00:00:00,0,32.602796,34.02,2013-06-16,32.897229
1,2013-06-16 01:00:00,1,28.765740,28.26,2013-06-16,29.090762
2,2013-06-16 02:00:00,2,25.175344,24.82,2013-06-16,25.497532
3,2013-06-16 03:00:00,3,23.792433,22.90,2013-06-16,24.148876
4,2013-06-16 04:00:00,4,23.438528,21.59,2013-06-16,23.868722
...,...,...,...,...,...,...
4435,2013-12-17 19:00:00,19,113.294493,113.92,2013-12-17,119.442217
4436,2013-12-17 20:00:00,20,109.309661,107.26,2013-12-17,107.770813
4437,2013-12-17 21:00:00,21,96.362564,89.02,2013-12-17,95.980939
4438,2013-12-17 22:00:00,22,79.490264,85.40,2013-12-17,83.808257


In [12]:
# start_date_q = "2025-01-01"
# end_date_q = "2025-12-31"
start_date_q = "2013-06-15"
end_date_q = "2013-12-18"

pred_cols_q = model_col

quantile_pi_dict = pi_quantile_mc(
    df_merged,
    start_date=start_date_q,
    end_date=end_date_q,
    N=14,
    true_col="true_lmp",
    pred_cols=pred_cols_q,
    quantiles=quantiles,
    M=20000,
    seed=123,
)

In [13]:
lear_result = quantile_pi_dict["Lear_lmp"]
dnn_result_copycing_lear_result = quantile_pi_dict["multitask_lmp"]
dnn_result = quantile_pi_dict["DNN_lmp"]

In [41]:
lear_result

,datetime,true,q10,q20,q30,q40,q50,q60,q70,q80,q90,q01,q02,q05,q95,q97,q99
0,2013-06-17 00:00:00,33.80,30.707910,31.336378,31.785829,32.167434,32.530123,32.896178,33.293529,33.753499,34.372230,29.221000,29.713075,30.183233,34.903175,35.334880,35.869119
1,2013-06-17 01:00:00,34.24,28.076183,28.296974,28.461111,28.602925,28.733530,28.862970,28.999157,29.159545,29.383807,27.541998,27.732635,27.886696,29.571046,29.738835,29.914437
2,2013-06-17 02:00:00,30.17,25.376624,25.531753,25.647091,25.746710,25.838485,25.929419,26.025108,26.137809,26.295394,25.001285,25.135244,25.243486,26.426932,26.544816,26.668203
3,2013-06-17 03:00:00,27.40,23.569861,23.959471,24.249121,24.499361,24.729844,24.958248,25.198553,25.481568,25.877319,22.627244,22.963647,23.235507,26.207742,26.503806,26.813666
4,2013-06-17 04:00:00,27.22,22.406860,23.213855,23.813826,24.332121,24.809563,25.282647,25.780404,26.366647,27.186394,20.454372,21.151213,21.714294,27.870776,28.484001,29.125847
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4411,2013-12-17 19:00:00,113.92,76.810613,89.322345,98.236419,105.992270,113.287378,120.379797,128.002159,137.129598,149.466913,48.403965,58.280942,66.884084,159.838985,168.987389,179.347456
4412,2013-12-17 20:00:00,107.26,65.501623,80.256461,91.169497,100.609817,109.322071,117.841799,127.136429,138.129416,152.874337,30.730048,42.298484,53.334229,165.580234,176.349505,189.166820
4413,2013-12-17 21:00:00,89.02,61.601081,73.524353,82.091662,89.488412,96.307494,103.002294,110.393199,119.388809,131.332667,32.793055,43.371041,51.893921,140.830089,149.565735,160.039711
4414,2013-12-17 22:00:00,85.40,58.507088,65.522255,70.838974,75.114839,79.321401,83.526532,88.028739,93.318786,100.571382,41.444888,47.387362,52.565757,106.622525,112.109463,117.797892


In [16]:
import numpy as np
import pandas as pd




test_set = [
    "2013-06-16",
    "2013-06-17",
    "2013-06-24",
    "2013-07-04",
    "2013-07-09",
    "2013-07-13",
    "2013-07-16",
    "2013-07-18",
    "2013-07-19",
    "2013-07-20",
    "2013-07-24",
    "2013-07-25",
    "2013-12-07",
    "2013-12-08",
    "2013-12-17"
]

benchmark_pl = [
    4.02875,
    7.97208,
    5.70395,
    12.15104,
    38.33541,
    44.22979,
    18.22395,
    31.56729,
    42.94958,
    2.85583,
    3.20395,
    22.38333
]



def avg_pinball_over_quantiles(df, y_col="true"):
    """
    Compute row-wise average pinball loss across
    q10, q20, ..., q90.
    """

    quantile_cols = [
        "q10", "q20", "q30", "q40", "q50",
        "q60", "q70", "q80", "q90"
    ]

    y_true = df[y_col].to_numpy()

    losses = []

    for col in quantile_cols:

        # q10 -> 0.10, q20 -> 0.20, ..., q90 -> 0.90
        q = int(col[1:]) / 100.0

        y_pred = df[col].to_numpy()

        error = y_true - y_pred

        loss_q = np.maximum(
            q * error,
            (q - 1) * error
        )

        losses.append(loss_q)

    # Shape:
    # rows x quantiles
    losses = np.stack(losses, axis=1)

    # Average across the 9 quantiles for each timestamp
    return np.mean(losses, axis=1)



lear_result_copy = lear_result.copy()
cing_lear_result_copy = cing_lear_result.copy()
dnn_result_copy = dnn_result.copy()

# Convert datetime
lear_result_copy["datetime"] = pd.to_datetime(
    lear_result_copy["datetime"]
)

# Create daily date
lear_result_copy["day"] = (
    lear_result_copy["datetime"]
    .dt.floor("D")
)

cing_lear_result_copy["datetime"] = pd.to_datetime(
    cing_lear_result_copy["datetime"]
)

# Create daily date
cing_lear_result_copy["day"] = (
    cing_lear_result_copy["datetime"]
    .dt.floor("D")
)


dnn_result_copy["datetime"] = pd.to_datetime(
    dnn_result_copy["datetime"]
)

# Create daily date
dnn_result_copy["day"] = (
    dnn_result_copy["datetime"]
    .dt.floor("D")
)



quantile_cols = [
    "q10", "q20", "q30", "q40", "q50",
    "q60", "q70", "q80", "q90"
]

required_cols = [
    "datetime",
    "true"
] + quantile_cols

missing_cols = [
    c for c in required_cols
    if c not in lear_result_copy.columns
]

if missing_cols:
    raise ValueError(
        f"Missing required columns: {missing_cols}"
    )



if lear_result_copy[
    ["true"] + quantile_cols
].isna().any().any():

    print(
        lear_result_copy[
            ["true"] + quantile_cols
        ].isna().sum()
    )

    raise ValueError(
        "Missing values found in truth or quantile predictions."


    )


if cing_lear_result_copy[
    ["true"] + quantile_cols
].isna().any().any():

    print(
        cing_lear_result_copy[
            ["true"] + quantile_cols
        ].isna().sum()
    )

    raise ValueError(
        "Missing values found in truth or quantile predictions."
    )



lear_result_copy["pinball_mean"] = (
    avg_pinball_over_quantiles(
        lear_result_copy,
        y_col="true"
    )
)

cing_lear_result_copy["pinball_mean"] = (
    avg_pinball_over_quantiles(
        cing_lear_result_copy,
        y_col="true"
    )
)

dnn_result_copy["pinball_mean"] = (
    avg_pinball_over_quantiles(
        dnn_result_copy,
        y_col="true"
    )
)



daily_pl_lear = (
    lear_result_copy
    .groupby(
        "day",
        as_index=False
    )["pinball_mean"]
    .mean()
)

daily_pl_cing_lear = (
    cing_lear_result_copy
    .groupby(
        "day",
        as_index=False
    )["pinball_mean"]
    .mean()
)

daily_pl_dnn = (
    dnn_result_copy
    .groupby(
        "day",
        as_index=False
    )["pinball_mean"]
    .mean()
)



benchmark_dates = pd.to_datetime(
    test_set[3:]
)

if len(benchmark_dates) != len(benchmark_pl):
    raise ValueError(
        "benchmark_dates and benchmark_pl must have the same length."
    )



benchmark_df = pd.DataFrame({
    "day": benchmark_dates,
    "benchmark_pl": benchmark_pl
})



test_pl_final_lear = benchmark_df.merge(
    daily_pl_lear,
    on="day",
    how="left"
)

test_pl_final_cing_lear = benchmark_df.merge(
    daily_pl_cing_lear,
    on="day",
    how="left"
)

test_pl_final_dnn = benchmark_df.merge(
    daily_pl_dnn,
    on="day",
    how="left"
)




test_pl_final_lear["ratings"] = (
    1
    - test_pl_final_lear["pinball_mean"]
    / test_pl_final_lear["benchmark_pl"]
)

test_pl_final_cing_lear["ratings"] = (
    1
    - test_pl_final_cing_lear["pinball_mean"]
    / test_pl_final_cing_lear["benchmark_pl"]
)

test_pl_final_dnn["ratings"] = (
    1
    - test_pl_final_dnn["pinball_mean"]
    / test_pl_final_dnn["benchmark_pl"]
)



print("Cing-Lear:")
display(test_pl_final_cing_lear)

print()
print(
    "Average rating:",
    test_pl_final_cing_lear["ratings"].mean()
)

print(
    "Average model pinball:",
    test_pl_final_cing_lear["pinball_mean"].mean()
)

print(
    "Average benchmark pinball:",
    test_pl_final_cing_lear["benchmark_pl"].mean()
)

print("Lear:")
display(test_pl_final_lear)

print()
print(
    "Average rating:",
    test_pl_final_lear["ratings"].mean()
)

print(
    "Average model pinball:",
    test_pl_final_lear["pinball_mean"].mean()
)

print(
    "Average benchmark pinball:",
    test_pl_final_lear["benchmark_pl"].mean()
)

print("DNN:")
display(test_pl_final_dnn)

print()
print(
    "Average rating:",
    test_pl_final_dnn["ratings"].mean()
)

print(
    "Average model pinball:",
    test_pl_final_dnn["pinball_mean"].mean()
)

print(
    "Average benchmark pinball:",
    test_pl_final_dnn["benchmark_pl"].mean()
)


Cing-Lear:


,day,benchmark_pl,pinball_mean,ratings
0,2013-07-04,4.02875,1.091898,0.728974
1,2013-07-09,7.97208,1.206345,0.848679
2,2013-07-13,5.70395,1.275044,0.776463
3,2013-07-16,12.15104,7.616522,0.373179
4,2013-07-18,38.33541,12.990869,0.661126
5,2013-07-19,44.22979,15.427812,0.651190
6,2013-07-20,18.22395,2.904392,0.840628
7,2013-07-24,31.56729,2.667862,0.915487
8,2013-07-25,42.94958,2.334804,0.945638
9,2013-12-07,2.85583,2.182661,0.235717



Average rating: 0.6753134218659526
Average model pinball: 4.613492498273057
Average benchmark pinball: 19.467079166666668
Lear:


,day,benchmark_pl,pinball_mean,ratings
0,2013-07-04,4.02875,1.210850,0.699448
1,2013-07-09,7.97208,1.244017,0.843953
2,2013-07-13,5.70395,1.127667,0.802301
3,2013-07-16,12.15104,7.157307,0.410972
4,2013-07-18,38.33541,13.313298,0.652715
5,2013-07-19,44.22979,16.911792,0.617638
6,2013-07-20,18.22395,2.985664,0.836168
7,2013-07-24,31.56729,2.586017,0.918079
8,2013-07-25,42.94958,2.412196,0.943837
9,2013-12-07,2.85583,2.211227,0.225715



Average rating: 0.6658457766815181
Average model pinball: 4.770118611766511
Average benchmark pinball: 19.467079166666668
DNN:


,day,benchmark_pl,pinball_mean,ratings
0,2013-07-04,4.02875,2.701864,0.329354
1,2013-07-09,7.97208,7.379104,0.074382
2,2013-07-13,5.70395,2.219309,0.610917
3,2013-07-16,12.15104,21.015230,-0.729501
4,2013-07-18,38.33541,22.291316,0.418519
5,2013-07-19,44.22979,42.043289,0.049435
6,2013-07-20,18.22395,12.552005,0.311236
7,2013-07-24,31.56729,5.602669,0.822517
8,2013-07-25,42.94958,5.104327,0.881155
9,2013-12-07,2.85583,3.277549,-0.147670



Average rating: 0.2617265213143707
Average model pinball: 11.15107523420115
Average benchmark pinball: 19.467079166666668


In [62]:

# # 每个模型导出一个 CSV：
# # 列顺序：datetime, true, q10, q20, ..., q90
# for model_col, df_q in quantile_pi_dict.items():
#     cols = ["datetime", "true"] + [f"q{int(q * 100):02d}" for q in quantiles]
#     df_out = df_q[cols]

#     fname = f"{model_col}_quantile_pi_{start_date_q}_{end_date_q}.csv"
#     df_out.to_csv(fname, index=False)
#     print(f"保存文件: {fname}，共 {len(df_out)} 行")


保存文件: Lear_lmp_quantile_pi_2025-01-01_2025-12-31.csv，共 8760 行
保存文件: multitask_lmp_quantile_pi_2025-01-01_2025-12-31.csv，共 8760 行
